In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [4]:
sam_sub = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [6]:
sam_sub.head()

,ID,Prediction
0,1,A B C
1,2,A B C
2,3,A B C
3,4,A B C
4,5,A B C


In [7]:
test.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [8]:
test.shape, sam_sub.shape, sam_sub.columns

((500, 7), (500, 2), Index(['ID', 'Prediction'], dtype='object'))

In [24]:
submission = pd.DataFrame({
    "ID": sam_sub["ID"], 
    "Prediction": sam_sub["Prediction"]
})

submission.to_csv("submission.csv", index=False)
print("Submission file saved!")

Submission file saved!


In [1]:
#trying WandB
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WB = user_secrets.get_secret("WB_KEY")


In [3]:
!pip install wandb -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 58.4 MB/s eta 0:00:00:00:0100:01


In [4]:
import wandb as wb

In [5]:
wb.login(WB)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

# Milestone 3

#### This milestone is designed to familiarize you with the core mechanics of Retrieval-Augmented Generation (RAG). You will learn how to query a vector index for relevant context and use a Cross-Encoder to re-rank the results for maximum accuracy. You will then run side-by-side A/B tests to compare a model's zero-shot inference with and without this context. Finally, we will explore the physical limits of context windows (tuning the correct number of chunks to retrieve) and demonstrate the catastrophic dangers of feeding an LLM incorrect data.

### Pre-Requisite code to run !

In [3]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


#### for q1, q2, q6

In [4]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
# 1. Run the zero-shot classifier on row 150
result = zs(prompt_150, candidate_labels=labels_150)

# 2. Map labels to their predicted probability scores
label_to_score = dict(zip(result['labels'], result['scores']))

# 3. Retrieve the score assigned to the ground-truth answer text
predicted_score = label_to_score[ans_150]

# 4. Display the result rounded to 3 decimal points
print(f"Predicted probability score for the ground-truth option: {predicted_score:.3f}")

Predicted probability score for the ground-truth option: 0.384


In [7]:
# 1. Embed the prompt for row index 150 using your variable 'model'
query_embedding = model.encode([prompt_150])

# 2. Query the FAISS index for the top k=10 results
distances, indices = index.search(query_embedding, 10)

# 3. Find the exact rank of the true correct document (index 150)
retrieved_indices = indices[0].tolist()

if 150 in retrieved_indices:
    # Adding 1 because list indices are 0-indexed, but ranks are 1-indexed
    rank = retrieved_indices.index(150) + 1
    print(f"The true correct document was placed at rank: {rank}")
else:
    print("The true correct document was not found in the top 10 retrieved documents.")

The true correct document was placed at rank: 10


In [8]:
# 1. Retrieve the unrelated document from your knowledge base index 999
wrong_context = kb[999]

# 2. Construct the Adversarial RAG prompt by injecting the wrong context
adversarial_prompt = f"""Use the following context to answer the question.
Context: {wrong_context}

Question: {prompt_150}"""

# 3. Run the zero-shot classifier on this poisoned string
adv_result = zs(adversarial_prompt, candidate_labels=labels_150)

# 4. Map the new labels to their adversarial probability scores
adv_label_to_score = dict(zip(adv_result['labels'], adv_result['scores']))

# 5. Extract the score assigned to the ground-truth answer text
adversarial_score = adv_label_to_score[ans_150]

print(f"Probability of the correct option under Adversarial RAG: {adversarial_score:.3f}")

Probability of the correct option under Adversarial RAG: 0.465


In [10]:
# 1. Initialize the Cross-Encoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# 2. Get the top 10 chunks using the retrieved_indices list from Q2
docs_10 = [kb[i] for i in retrieved_indices]

# 3. Create prompt-context pairs and get the Cross-Encoder relevance scores
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# 4. Pair the original KB indices with their new Cross-Encoder scores and sort them
# We zip retrieved_indices with ce_scores so we can track where document 150 ends up
reranked_results = sorted(zip(ce_scores, retrieved_indices), key=lambda x: x[0], reverse=True)

# 5. Extract just the sorted indices to determine the new rank
reranked_indices = [idx for score, idx in reranked_results]

# 6. Find the exact rank (1-indexed) of the true correct document (index 150)
if 150 in reranked_indices:
    ce_rank = reranked_indices.index(150) + 1
    print(f"The Cross-Encoder placed the true correct document at rank: {ce_rank}")
else:
    print("The true correct document is not in the reranked list.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The Cross-Encoder placed the true correct document at rank: 1


In [11]:
from transformers import AutoTokenizer

# 1. Initialize the bert-base-uncased tokenizer
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 2. Get the prompt for row index 42
prompt_42 = str(train.iloc[42]['prompt']) 

# 3. Embed prompt 42 and query your FAISS index for the top k=5 documents
query_embedding_42 = model.encode([prompt_42])
distances_42, indices_42 = index.search(query_embedding_42, 5)
retrieved_indices_42 = indices_42[0].tolist()

# 4. Extract the top 5 documents and concatenate them with a single space
docs_5 = [kb[i] for i in retrieved_indices_42]
concatenated_docs = " ".join(docs_5)

# 5. Create the required text format
formatted_string = f"Context: {concatenated_docs} Question: {prompt_42}"

# 6. Tokenize without truncation and calculate the exact total tokens
tokens_42 = bert_tokenizer(formatted_string, truncation=False)
total_tokens = len(tokens_42['input_ids'])

print(f"Total tokens generated: {total_tokens}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Total tokens generated: 216


In [12]:
# 1. Retrieve the exact true document for row index 150 from your KB
true_doc_150 = kb[150]

# 2. Create the RAG string as specified
rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"

# 3. Run the zero-shot classifier on this augmented string
rag_result = zs(rag_string_150, candidate_labels=labels_150)

# 4. Map the new labels to their predicted probability scores
rag_label_to_score = dict(zip(rag_result['labels'], rag_result['scores']))

# 5. Extract the score assigned to the ground-truth answer text
new_predicted_score = rag_label_to_score[ans_150]

# 6. Print the result rounded to 3 decimal places
print(f"New predicted probability score with true context: {new_predicted_score:.3f}")

New predicted probability score with true context: 0.989


In [13]:
# 1. Retrieve the unrelated document from your knowledge base index 999
wrong_context = kb[999]

# 2. Construct the Adversarial RAG string matching your format preference
adversarial_string = f"Context: {wrong_context} Question: {prompt_150}"

# 3. Run the zero-shot classifier on this poisoned string
adv_result = zs(adversarial_string, candidate_labels=labels_150)

# 4. Map the new labels to their adversarial probability scores
adv_label_to_score = dict(zip(adv_result['labels'], adv_result['scores']))

# 5. Extract the score assigned to the ground-truth answer text
adversarial_score = adv_label_to_score[ans_150]

print(f"Probability of the correct option under Adversarial RAG: {adversarial_score:.3f}")

Probability of the correct option under Adversarial RAG: 0.529


In [14]:
hit_count = 0
total_rows = 100

for i in range(total_rows):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    
    # Extract the exact text string of the correct answer option
    correct_option_letter = row['answer']
    correct_text_string = str(row[correct_option_letter])
    
    # 1. Embed the prompt
    query_embedding = model.encode([prompt])
    
    # 2. Retrieve top k=5 documents from FAISS
    distances, indices = index.search(query_embedding, 5)
    retrieved_indices = indices[0].tolist()
    retrieved_docs = [kb[idx] for idx in retrieved_indices if idx < len(kb)]
    
    # 3. Check for a "Hit" (if the exact text string is inside any retrieved document)
    is_hit = any(correct_text_string in doc for doc in retrieved_docs)
    
    if is_hit:
        hit_count += 1

# 4. Calculate the Hit Rate percentage
hit_rate_percentage = (hit_count / total_rows) * 100

print(f"Exact Hit Rate percentage: {hit_rate_percentage:.1f}%")

Exact Hit Rate percentage: 73.0%


In [15]:
# Function to calculate MAP@3 for a single row
def calculate_map3(predictions, ground_truth):
    """
    predictions: list of top 3 predicted letters, e.g., ['C', 'A', 'E']
    ground_truth: the single correct letter string, e.g., 'A'
    """
    score = 0.0
    for rank, pred in enumerate(predictions):
        if pred == ground_truth:
            # Formula for MAP@3 when there is exactly 1 correct answer: 1 / (rank + 1)
            score = 1.0 / (rank + 1)
            break
    return score

# Initialize lists to store individual MAP@3 scores
map3_scores = []
total_eval_rows = 20

print("Processing evaluation loop...")

for i in range(total_eval_rows):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    ground_truth_letter = str(row['answer'])
    
    # Extract the mapping between label text strings and their option letters
    # This allows us to map the classifier's text predictions back to 'A', 'B', 'C', 'D', 'E'
    text_to_letter = {str(row[let]): let for let in ['A', 'B', 'C', 'D', 'E']}
    candidate_labels = list(text_to_letter.keys())
    
    # 1. RETRIEVE: Get top k=5 documents from FAISS
    query_embedding = model.encode([prompt])
    distances, indices = index.search(query_embedding, 5)
    retrieved_indices = indices[0].tolist()
    retrieved_docs = [kb[idx] for idx in retrieved_indices if idx < len(kb)]
    
    # 2. RERANK: Evaluate the 5 documents with the Cross-Encoder
    ce_pairs = [[prompt, doc] for doc in retrieved_docs]
    ce_scores = cross_encoder.predict(ce_pairs)
    
    # Select the single document text with the absolute highest score
    best_doc_idx = np.argmax(ce_scores)
    best_document = retrieved_docs[best_doc_idx]
    
    # 3. AUGMENT: Format the RAG string exactly as specified
    rag_string = f"Context: {best_document} Question: {prompt}"
    
    # 4. PREDICT: Pass the string to the zero-shot classifier
    result = zs(rag_string, candidate_labels=candidate_labels)
    
    # 5. SCORE: Sort option letters by highest probability to lowest
    # result['labels'] arrives pre-sorted by highest probability score from the pipeline
    top_3_letters = [text_to_letter[label] for label in result['labels'][:3]]
    
    # Calculate MAP@3 for the current row
    row_map3 = calculate_map3(top_3_letters, ground_truth_letter)
    map3_scores.append(row_map3)

# Calculate final average MAP@3 score
final_average_map3 = np.mean(map3_scores)

print(f"\nFinal Average MAP@3 Score: {final_average_map3:.3f}")

Processing evaluation loop...

Final Average MAP@3 Score: 0.975


In [10]:
import torch
print(torch.__version__)

2.10.0+cpu


In [11]:
!pip install -q transformers datasets peft accelerate

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMultipleChoice

In [12]:
# 1. Load Kaggle competition dataset
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# 2. Label mappings (A=0, B=1, C=2, D=3, E=4)
label_to_index = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
index_to_label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

print(f"Dataset loaded successfully with {len(train)} rows.")

Dataset loaded successfully with 2000 rows.


In [13]:
# Model Checkpoints
DEBERTA_MODEL_NAME = "microsoft/deberta-v3-small"
ROBERTA_MODEL_NAME = "roberta-base"

# Initialize Tokenizers
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL_NAME)
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL_NAME)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
def prepare_mc_inputs(df, tokenizer, max_length=128):
    """
    Formats and tokenizes multiple-choice inputs for AutoModelForMultipleChoice.
    """
    all_input_ids = []
    all_attention_masks = []
    options = ['A', 'B', 'C', 'D', 'E']
    
    for _, row in df.iterrows():
        prompt = str(row['prompt'])
        choices = [prompt + " " + (tokenizer.sep_token or "[SEP]") + " " + str(row[opt]) for opt in options]
        
        tokenized = tokenizer(
            choices,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        
        all_input_ids.append(tokenized['input_ids'])
        all_attention_masks.append(tokenized['attention_mask'])
        
    input_ids = torch.stack(all_input_ids)           # Shape: [N, 5, max_length]
    attention_mask = torch.stack(all_attention_masks) # Shape: [N, 5, max_length]
    
    return input_ids, attention_mask